- 3 subt params
- 2 pf params

In [83]:
import numpy as np
import scipy as sp
from matplotlib import pyplot as plt

import pandas as pd
import seaborn as sns
import os
from statsmodels.stats.multitest import fdrcorrection
# from google.colab import drive
# drive.mount('/content/drive')

from scipy.stats import zscore

import statsmodels.api as sm

from pymer4.models import Lmer, Lm

# !pip install scikit-bio

In [84]:
# mixed_batch_data = pd.read_csv('/content/drive/MyDrive/curvefit_traits_merged_linearpf_onlyneccols.csv', index_col='subID')
mixed_batch_data = pd.read_csv('../../../data/subtlety_playfight_data/curvefit_traits_merged_linearpf_onlyneccols_fight.csv', index_col='subID')
mixed_batch_data.drop('Unnamed: 0', axis=1, inplace=True)
mixed_batch_data

,PSE_subt,range_subt,bias_subt,sigma_subt,PSE_pf,slope_pf,range_pf,bias_pf,social_skill,attn_switch,...,comm,posAffect,negAffect,neuroticism,extraversion,openness,agreeableness,conscientiousness,loneliness,nfriends
subID,,,,,,,,,,,,,,,,,,,,,
30002.0,0.552951,0.900821,0.315504,0.103360,0.513054,0.784286,0.784286,0.452539,21.0,23.0,...,16.000000,20.0,0.0,38.0,28.000000,31.000000,34.0,39.0,56.0,3.0
30004.0,0.276896,0.805834,0.999458,0.076866,0.872907,0.469286,0.469286,0.170256,21.0,20.0,...,15.000000,25.0,11.0,38.0,26.181818,50.000000,50.0,46.0,56.0,3.0
30005.0,0.320583,0.710131,0.861730,0.115353,0.501696,0.842143,0.842143,0.490950,17.0,29.0,...,16.000000,20.0,21.0,51.0,39.000000,40.000000,43.0,19.0,41.0,5.0
30006.0,0.464823,0.989396,0.889572,0.079221,0.741046,0.674587,0.674587,0.000307,8.0,10.0,...,3.000000,25.0,0.0,18.0,41.000000,31.000000,44.0,57.0,30.0,4.0
30008.0,0.325118,0.582252,0.692710,0.172585,0.677591,0.511071,0.511071,0.314366,5.0,15.0,...,12.222222,28.0,4.0,27.0,46.000000,42.545455,46.0,50.0,45.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30344.0,0.361233,0.655715,0.794480,0.135840,0.492859,0.566786,0.566786,0.509343,27.0,18.0,...,16.000000,14.0,0.0,25.0,29.000000,49.000000,35.0,56.0,41.0,6.0
30345.0,0.475067,0.833333,1.000000,0.013915,0.639965,0.781136,0.781136,0.000457,15.0,14.0,...,5.000000,15.0,15.0,36.0,23.000000,37.000000,45.0,54.0,72.0,1.0
30346.0,0.426850,0.999914,0.961644,0.045409,0.412504,0.786429,0.786429,0.822185,16.0,25.0,...,8.000000,27.0,6.0,49.0,36.000000,36.000000,43.0,44.0,56.0,3.0


In [85]:
mixed_batch_data.isna().sum()

PSE_subt             0
range_subt           0
bias_subt            0
sigma_subt           0
PSE_pf               0
slope_pf             0
range_pf             0
bias_pf              0
social_skill         0
attn_switch          0
img                  0
attn_to_det          0
comm                 0
posAffect            0
negAffect            0
neuroticism          0
extraversion         0
openness             0
agreeableness        0
conscientiousness    0
loneliness           0
nfriends             1
dtype: int64

Looks like one subject is missing a value for `nfriends`. Let's just drop them as this will cause headaches later

In [86]:
mixed_batch_data = mixed_batch_data.dropna()
mixed_batch_data.shape

(274, 22)

In [87]:
mixed_batch_data.columns

Index(['PSE_subt', 'range_subt', 'bias_subt', 'sigma_subt', 'PSE_pf',
       'slope_pf', 'range_pf', 'bias_pf', 'social_skill', 'attn_switch', 'img',
       'attn_to_det', 'comm', 'posAffect', 'negAffect', 'neuroticism',
       'extraversion', 'openness', 'agreeableness', 'conscientiousness',
       'loneliness', 'nfriends'],
      dtype='object')

In [88]:
curve_param_cols = [
                  'PSE_subt',
                  'range_subt',
                  'bias_subt',
                  'range_pf',
                  'bias_pf'
                    ]
subt_param_cols = [
                  'PSE_subt',
                  'range_subt',
                  'bias_subt',
                   ]
pf_param_cols = [
                 'range_pf',
                 'bias_pf'
                 ]
trait_cols = [
              'social_skill', 'attn_switch', 'img', 'attn_to_det', 'comm',
              'posAffect',
              'negAffect',
              'neuroticism', 'extraversion', 'openness', 'agreeableness', 'conscientiousness',
              'loneliness',
              'nfriends'
       ]

social_cols = [
              'social_skill', 'attn_switch', 'img', 'attn_to_det', 'comm',
              'loneliness',
              'nfriends'
              ]

In [89]:
results_loc= '../../../results/subtlety_playfight_mixed/trait-beh/extra_analyses/'

In [90]:
# bias_PSE_subt = mixed_batch_data['bias_subt']/ mixed_batch_data['PSE_subt']
# bias_PSE_pf = mixed_batch_data['bias_pf']/ mixed_batch_data['PSE_pf']
# mixed_batch_data.insert(3, 'bias_PSE_subt', bias_PSE_subt)
# mixed_batch_data.insert(9, 'bias_PSE_pf', bias_PSE_pf)
# mixed_batch_data

In [91]:
# first, super important to z-score all the features in order to minimize the impact of different features being measured on different scales (just like PCA)
mixed_batch_data_zscored = mixed_batch_data.apply(zscore)
mixed_batch_data_zscored

,PSE_subt,range_subt,bias_subt,sigma_subt,PSE_pf,slope_pf,range_pf,bias_pf,social_skill,attn_switch,...,comm,posAffect,negAffect,neuroticism,extraversion,openness,agreeableness,conscientiousness,loneliness,nfriends
subID,,,,,,,,,,,,,,,,,,,,,
30002.0,1.587100,1.139946,-2.182466,0.144545,-0.206179,0.740744,0.802900,0.052813,1.249255,1.403779,...,1.044306,-0.214096,-1.031925,0.328778,-1.032702,-1.533105,-1.890716,-0.771265,0.884033,-0.304004
30004.0,-0.836964,0.619923,1.390323,-0.452518,2.089832,-0.732085,-0.731401,-0.970100,1.249255,0.813345,...,0.862296,0.379919,0.470275,0.328778,-1.249254,1.402822,1.569992,0.078101,0.884033,-0.304004
30005.0,-0.453342,0.095984,0.670868,0.414827,-0.278646,1.011263,1.084710,0.192006,0.604630,2.584649,...,1.044306,-0.214096,1.835912,1.543459,0.277433,-0.142403,0.055932,-3.198024,-0.300924,0.449816
30006.0,0.813243,1.624863,0.816307,-0.399448,1.248505,0.227830,0.268579,-1.585944,-0.845776,-1.154772,...,-1.321823,0.379919,-1.031925,-1.539962,0.515640,-1.533105,0.272226,1.412818,-1.169892,0.072906
30008.0,-0.413524,-0.604109,-0.212045,1.704615,0.843637,-0.536710,-0.527871,-0.447886,-1.329245,-0.170714,...,0.356713,0.736328,-0.485670,-0.699029,1.111156,0.250927,0.704815,0.563453,0.015065,0.449816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30344.0,-0.096389,-0.201924,0.319575,0.876526,-0.335034,-0.276210,-0.256498,0.258657,2.216193,0.419721,...,1.044306,-0.926914,-1.031925,-0.885903,-0.913599,1.248300,-1.674422,1.291480,-0.300924,0.826726
30345.0,0.903194,0.770473,1.393153,-1.871185,0.603566,0.726018,0.787560,-1.585402,0.282318,-0.367525,...,-0.957803,-0.808111,1.016530,0.141904,-1.628218,-0.605970,0.488521,1.048804,2.147987,-1.057824
30346.0,0.479799,1.682444,1.192794,-1.161440,-0.847731,0.750763,0.813338,1.392306,0.443474,1.797403,...,-0.411773,0.617525,-0.212543,1.356585,-0.079876,-0.760493,0.055932,-0.164575,0.884033,-0.304004


In [92]:
mixed_batch_data['PSE_subt'].describe()

count    274.000000
mean       0.372210
std        0.114089
min        0.000000
25%        0.327620
50%        0.387369
75%        0.444022
max        0.729328
Name: PSE_subt, dtype: float64

In [93]:
bias_PSE_subt = mixed_batch_data_zscored['bias_subt']/ mixed_batch_data_zscored['PSE_subt']
bias_PSE_pf = mixed_batch_data_zscored['bias_pf']/ mixed_batch_data_zscored['PSE_pf']

bias_PSE_subt_diff = mixed_batch_data_zscored['bias_subt'] - mixed_batch_data_zscored['PSE_subt']
bias_PSE_pf_diff = mixed_batch_data_zscored['bias_pf'] - mixed_batch_data_zscored['PSE_pf']

# bias_PSE_subt = zscore(mixed_batch_data_zscored['bias_subt']/ mixed_batch_data_zscored['PSE_subt'])
# bias_PSE_pf = zscore(mixed_batch_data_zscored['bias_pf']/ mixed_batch_data_zscored['PSE_pf'])

mixed_batch_data_zscored.insert(3, 'bias_PSE_subt', bias_PSE_subt)
mixed_batch_data_zscored.insert(4, 'bias_PSE_subt_diff', bias_PSE_subt_diff)
mixed_batch_data_zscored.insert(9, 'bias_PSE_pf', bias_PSE_pf)
mixed_batch_data_zscored.insert(10, 'bias_PSE_pf_diff', bias_PSE_pf_diff)
mixed_batch_data_zscored

,PSE_subt,range_subt,bias_subt,bias_PSE_subt,bias_PSE_subt_diff,sigma_subt,PSE_pf,slope_pf,range_pf,bias_PSE_pf,...,comm,posAffect,negAffect,neuroticism,extraversion,openness,agreeableness,conscientiousness,loneliness,nfriends
subID,,,,,,,,,,,,,,,,,,,,,
30002.0,1.587100,1.139946,-2.182466,-1.375128,-3.769566,0.144545,-0.206179,0.740744,0.802900,-0.256151,...,1.044306,-0.214096,-1.031925,0.328778,-1.032702,-1.533105,-1.890716,-0.771265,0.884033,-0.304004
30004.0,-0.836964,0.619923,1.390323,-1.661150,2.227287,-0.452518,2.089832,-0.732085,-0.731401,-0.464200,...,0.862296,0.379919,0.470275,0.328778,-1.249254,1.402822,1.569992,0.078101,0.884033,-0.304004
30005.0,-0.453342,0.095984,0.670868,-1.479827,1.124210,0.414827,-0.278646,1.011263,1.084710,-0.689067,...,1.044306,-0.214096,1.835912,1.543459,0.277433,-0.142403,0.055932,-3.198024,-0.300924,0.449816
30006.0,0.813243,1.624863,0.816307,1.003768,0.003064,-0.399448,1.248505,0.227830,0.268579,-1.270275,...,-1.321823,0.379919,-1.031925,-1.539962,0.515640,-1.533105,0.272226,1.412818,-1.169892,0.072906
30008.0,-0.413524,-0.604109,-0.212045,0.512777,0.201478,1.704615,0.843637,-0.536710,-0.527871,-0.530899,...,0.356713,0.736328,-0.485670,-0.699029,1.111156,0.250927,0.704815,0.563453,0.015065,0.449816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30344.0,-0.096389,-0.201924,0.319575,-3.315470,0.415964,0.876526,-0.335034,-0.276210,-0.256498,-0.772031,...,1.044306,-0.926914,-1.031925,-0.885903,-0.913599,1.248300,-1.674422,1.291480,-0.300924,0.826726
30345.0,0.903194,0.770473,1.393153,1.542475,0.489960,-1.871185,0.603566,0.726018,0.787560,-2.626725,...,-0.957803,-0.808111,1.016530,0.141904,-1.628218,-0.605970,0.488521,1.048804,2.147987,-1.057824
30346.0,0.479799,1.682444,1.192794,2.486030,0.712995,-1.161440,-0.847731,0.750763,0.813338,-1.642392,...,-0.411773,0.617525,-0.212543,1.356585,-0.079876,-0.760493,0.055932,-0.164575,0.884033,-0.304004


# predict curve fit params from all traits

In [94]:
trait_cols


['social_skill',
 'attn_switch',
 'img',
 'attn_to_det',
 'comm',
 'posAffect',
 'negAffect',
 'neuroticism',
 'extraversion',
 'openness',
 'agreeableness',
 'conscientiousness',
 'loneliness',
 'nfriends']

In [95]:
param_list = ['bias_PSE_subt','bias_PSE_subt_diff','bias_subt','PSE_subt']

all_ps = np.full((len(trait_cols),len(param_list)),np.nan) # for MCC
for iparam,param in enumerate(param_list):
    print(f'\nPARAM {param}')
    model = Lm(f"{param} ~ social_skill + attn_switch + img + attn_to_det + comm + posAffect + negAffect + neuroticism + extraversion + openness + agreeableness + conscientiousness + loneliness + nfriends", data=mixed_batch_data_zscored)
    print(model.fit())
    for t,trait in enumerate(trait_cols): # for MCC
        all_ps[t, iparam] = model.coefs['P-val'][trait]


PARAM bias_PSE_subt
Formula: bias_PSE_subt~social_skill+attn_switch+img+attn_to_det+comm+posAffect+negAffect+neuroticism+extraversion+openness+agreeableness+conscientiousness+loneliness+nfriends

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.032	 R^2_adj: -0.020

Log-likelihood: -1480.622 	 AIC: 2991.244	 BIC: 3045.441

Fixed effects:

                   Estimate  2.5_ci  97.5_ci     SE   DF  T-stat  P-val Sig
Intercept            -4.088 -10.667    2.492  3.341  259  -1.223  0.222    
social_skill          1.879 -12.936   16.694  7.524  259   0.250  0.803    
attn_switch           2.567  -7.616   12.749  5.171  259   0.496  0.620    
img                  -1.094  -9.577    7.389  4.308  259  -0.254  0.800    
attn_to_det          -2.196  -9.710    5.318  3.816  259  -0.575  0.566    
comm                 -1.060 -13.261   11.140  6.196  259  -0.171  0.864    
posAffect            -0.258 -10.651   10

In [96]:
param_list = ['bias_PSE_pf','bias_PSE_pf_diff', 'bias_pf','PSE_pf']

all_ps = np.full((len(trait_cols),len(param_list)),np.nan) # for MCC
for iparam,param in enumerate(param_list):
    print(f'\nPARAM {param}')
    model = Lm(f"{param} ~ social_skill + attn_switch + img + attn_to_det + comm + posAffect + negAffect + neuroticism + extraversion + openness + agreeableness + conscientiousness + loneliness + nfriends", data=mixed_batch_data_zscored)
    print(model.fit())
    for t,trait in enumerate(trait_cols): # for MCC
        all_ps[t, iparam] = model.coefs['P-val'][trait]


PARAM bias_PSE_pf
Formula: bias_PSE_pf~social_skill+attn_switch+img+attn_to_det+comm+posAffect+negAffect+neuroticism+extraversion+openness+agreeableness+conscientiousness+loneliness+nfriends

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.085	 R^2_adj: 0.036

Log-likelihood: -1284.816 	 AIC: 2599.633	 BIC: 2653.830

Fixed effects:

                   Estimate  2.5_ci  97.5_ci     SE   DF  T-stat  P-val Sig
Intercept            -2.413  -5.633    0.807  1.635  259  -1.476  0.141    
social_skill         -0.452  -7.702    6.798  3.682  259  -0.123  0.902    
attn_switch          -1.808  -6.791    3.175  2.531  259  -0.715  0.476    
img                  -2.536  -6.687    1.616  2.108  259  -1.203  0.230    
attn_to_det           0.633  -3.044    4.311  1.867  259   0.339  0.735    
comm                  3.772  -2.199    9.742  3.032  259   1.244  0.215    
posAffect             0.778  -4.308    5.864 

In [66]:
result_bool = np.full_like(all_ps,np.nan)
corrected_p = np.full_like(all_ps,np.nan)
for t, trait in enumerate(trait_cols): # MCC one PC at a time
    result_bool[t,:], corrected_p[t,:] = fdrcorrection(all_ps[t,:]) # MCC across all params for one PC (regressor) at a time
    print(trait, np.where(result_bool[t,:])[0], np.array(param_list)[np.where(result_bool[t,:])[0]])

social_skill [] []
attn_switch [] []
img [] []
attn_to_det [] []
comm [] []
posAffect [] []
negAffect [] []
neuroticism [] []
extraversion [] []
openness [0] ['bias_PSE_pf']
agreeableness [] []
conscientiousness [] []
loneliness [1 2] ['bias_pf' 'PSE_pf']
nfriends [] []


In [109]:
from scipy import stats

In [110]:
x = mixed_batch_data_zscored['comm']
y = mixed_batch_data_zscored['bias_subt']
stats.pearsonr(x,y)

PearsonRResult(statistic=np.float64(-0.12728067802299373), pvalue=np.float64(0.03522083935901831))

In [113]:
x = mixed_batch_data_zscored['agreeableness']
y = mixed_batch_data_zscored['PSE_subt']
stats.pearsonr(x,y)

PearsonRResult(statistic=np.float64(0.0921577530255825), pvalue=np.float64(0.1280721402383504))